# Lensing Map EDA #

### In this notebook, we examine weak lensing maps to identify where values are defined and undefined, as well as visualize weak lensing maps for the DC2 dataset ###

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
os.chdir('/home/shreyasc/bliss') # change this to your path

In [ ]:
import torch
import numpy as np
from os import environ
from pathlib import Path
from einops import rearrange
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd

from hydra import initialize, compose
from hydra.utils import instantiate
import healpy as hp
import math
import pathlib

import numpy as np
import pandas as pd
import torch
from astropy.io import fits
from astropy.io.fits import Header
from astropy.wcs import WCS
from einops import rearrange
from torch.utils.data import DataLoader, Dataset
from tqdm import trange
from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib.colors import LinearSegmentedColormap
from bliss.surveys.dc2 import read_frame_for_band
import pickle

IMPORTANT: If you don't need to regenerate / load all the data and just want to visualize the maps, skip to the section titled "Load Previously Generated Data and Visualize Maps" and do not run the next few cells

In [ ]:
data_dir = "/data/scratch/dc2local/run2.2i-dr6-v4/coadd-t3828-t3829/deepCoadd-results/"
image_lim = [4000, 4000]

In [ ]:
BANDS = ("g", "i", "r", "u", "y", "z")

In [ ]:
# extracted method from DC2 class, identical but provides minimal required functionality
def load_image_and_bg_files_list(bands, data_dir):
        img_pattern = "**/*/calexp*.fits"
        bg_pattern = "**/*/bkgd*.fits"
        image_files = []
        bg_files = []

        for band in bands:
            band_path = data_dir + str(band)
            img_file_list = list(pathlib.Path(band_path).glob(img_pattern))
            bg_file_list = list(pathlib.Path(band_path).glob(bg_pattern))

            image_files.append(sorted(img_file_list))
            bg_files.append(sorted(bg_file_list))
        n_image = len(bg_files[0])

        return n_image, image_files, bg_files

In [ ]:
n_image, image_files, bg_files = load_image_and_bg_files_list(BANDS, data_dir)

In [ ]:
image_lists = []
bg_lists = []
wcs_lists = []
for img_idx in trange(n_image):
    imlist, bglist, wcs = read_frame_for_band(image_files, bg_files, img_idx, len(BANDS), image_lim)
    image_lists.append(imlist)
    bg_lists.append(bglist)
    wcs_lists.append(WCS(Header.fromstring(wcs)))

NOTE: do not run the below cells unless you want to regenerate and save the entire shear / convergence map dataset (currently takes > 2 hours to load and save the entire data)

instead, consider skipping to the section titled "Load Previously Generated Data and Visualize Maps"

In [ ]:
# now we can load the catalog
catalog = pd.read_csv("/data/scratch/shreyasc/cosmo_only.csv")

In [ ]:
catalog

In [ ]:
# each value in the catalog is a unique truth galaxy
catalog["galaxy_id"].isna().sum()

In [ ]:
# verifying the above again -- every value is unique
len(catalog["galaxy_id"].unique())

In [ ]:
truth_cat = pd.read_csv('/data/scratch/shreyasc/truth_data_only.csv')
truth_ids = truth_cat["cosmodc2_id"]
cosmo_ids = catalog["galaxy_id"]
result = cosmo_ids[~cosmo_ids.isin(truth_ids)]
result

In [ ]:
truth_cat

In [ ]:
img_ids = []
shear_1s = []
shear_2s = []
plocss = []

# respective location to the first image (our "absolute location") to put maps in perspective
respective_im_0_plocss = []
wcs_0 = wcs_lists[0]

ra = torch.tensor(catalog["ra"].values).numpy().squeeze()
dec = torch.tensor(catalog["dec"].values).numpy().squeeze()

truth_catalog_ra = torch.tensor(truth_cat["ra"].values).numpy().squeeze()
truth_catalog_dec = torch.tensor(truth_cat["dec"].values).numpy().squeeze()

shear_1 = torch.tensor(catalog["shear_1"].values)
shear_2 = torch.tensor(catalog["shear_2"].values)
for imidx in trange(len(wcs_lists)):
    wcs = wcs_lists[imidx]
    img_list = image_lists[imidx]
    plocs_lim = img_list[0].shape
    height = plocs_lim[0]
    width = plocs_lim[1]
    pt, pr = wcs.all_world2pix(ra, dec, 0)  # convert to pixel coordinates
    plocs = torch.stack((torch.tensor(pr), torch.tensor(pt)), dim=-1)

    plocs = (plocs.reshape(1, plocs.size()[0], 2))[0]

    # add another plocs thing for the galaxies taken from truth_data_only
    # compare those plocs to wcs from image
    # store galaxy locations of current image
    # plot galaxies overlaid on image if we can
    # alternatively we can overlay the map over the actual image (with galaxy)

    respective_im_0_pt, respective_im_0_pr = wcs_0.all_world2pix(ra, dec, 0)
    respective_im_0_plocs = torch.stack((torch.tensor(respective_im_0_pr), torch.tensor(respective_im_0_pt)), dim=-1)
    respective_im_0_plocs = (respective_im_0_plocs.reshape(1, respective_im_0_plocs.size()[0], 2))[0]
    
    # mask for current patch
    x0_mask = (plocs[:, 0] > 0) & (plocs[:, 0] < width)
    x1_mask = (plocs[:, 1] > 0) & (plocs[:, 1] < height)
    x_mask = x0_mask * x1_mask
    
    shear_1s.append(shear_1[x_mask])
    shear_2s.append(shear_2[x_mask])
    plocss.append(plocs[x_mask])
    respective_im_0_plocss.append(respective_im_0_plocs[x_mask])

In [ ]:

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/image_lists.pkl", "wb") as f:
    pickle.dump(image_lists, f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/shear_1.pkl", "wb") as f:
    pickle.dump(shear_1s, f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/shear_2.pkl", "wb") as f:
    pickle.dump(shear_2s, f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/plocs.pkl", "wb") as f:
    pickle.dump(plocss, f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/im0_plocs.pkl", "wb") as f:
    pickle.dump(respective_im_0_plocss, f)

### Load Previously Generated Data and Visualize Maps ###

In [ ]:
with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/image_lists.pkl", "rb") as f:
    image_lists = pickle.load(f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/shear_1.pkl", "rb") as f:
    shear_1s = pickle.load(f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/shear_2.pkl", "rb") as f:
    shear_2s = pickle.load(f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/plocs.pkl", "rb") as f:
    plocss = pickle.load(f)

with open("/nfs/turbo/lsa-regier/scratch/shreyasc/dc2_lensing_eda/im0_plocs.pkl", "rb") as f:
    respective_im_0_plocss = pickle.load(f)

In [ ]:
num_shears = [len(shear_1) for shear_1 in shear_1s] # full image
print("Mean shear data points per image: ", np.mean(num_shears))
print("Standard dev shear data points per image: ", np.std(num_shears))
print("Maximum shears:", max(num_shears))
print("Minimum shears:", min(num_shears))

In [ ]:
# Definitely not normally distributed...
n, bins, patches = plt.hist(num_shears, bins=20)
plt.show()

Run the three cells below to generate plots for horizontal and diagonal shear. Note that at present, it takes ~30 seconds to generate a single plot due to the calculations

In [ ]:
def plot_single_shear_map(idx, grid_size, original_size, shear_type, show_zeros=False, image_overlay=False):
    image_list, s1, s2, ploc = image_lists[idx], shear_1s[idx], shear_2s[idx], plocss[idx]
    image = np.stack(image_list) # for printing image
    image = np.sum(image, axis=0)
    vmin = image.min().item()
    vmax = image.max().item()

    # Create a grid to represent the shear field
    shear_grid = np.zeros(grid_size)  # Initialize grid with zeros
    num_shears_per_grid_loc = np.zeros(grid_size, dtype=float) # + math.ulp(1.0) # denominator for average shear in grid locations, added epsilon due to possible divide by zero if no estimates in a given grid location

    sum_x_sq = np.zeros(grid_size) # running total of sums of squared x (for use in stdev calculations)

    # Assign shear values to the corresponding grid cells
    if shear_type == "Horizontal Shear":
        shear_og = s1
    elif shear_type == "Diagonal Shear":
        shear_og = s2
    else:
        print("Unknown shear type: ", shear_type, " not in {'Horizontal Shear', 'Diagonal Shear'}")
    for inner_idx, (pixel1, pixel2) in enumerate(ploc):
        # TODO: replace the below with an accumulator (torch.gather or torch.scatter(?)) since currently it replaces
        i = int((pixel1 / float(original_size[0])) * (grid_size[0] - 1))  # Scale pixel coordinate to fit the grid
        j = int((pixel2 / float(original_size[1])) * (grid_size[1] - 1))  # Scale pixel coordinate to fit the grid
        shear_grid[i, j] += shear_og[inner_idx] # accumulator for shear in grid location
        sum_x_sq[i, j] += (shear_og[inner_idx]**2) # add square to accumulator for stdev calculation
        num_shears_per_grid_loc += 1.0 # accumulator for denominator of grid locations

    shear_grid /= num_shears_per_grid_loc # average across grid size
    shear_grid *= (10**6) # scale up from 1e-6
    n = (original_size[0] / grid_size[0]) * (original_size[1] / grid_size[1]) # number of locations in each grid
    sum_x_sq *= (10**6)
    grid_stdev = np.sqrt((sum_x_sq / n) - (shear_grid ** 2))
    print("Mean SD of shear value dispersion per grid point, including zeros: ", np.mean(grid_stdev))

    print("X: ", "(" + str(torch.min(respective_im_0_plocss[idx][:,0]).item()) + "," + str(torch.max(respective_im_0_plocss[idx][:,0]).item()) + ")", "Y: (" + str(torch.min(respective_im_0_plocss[idx][:,1]).item()) + "," + str(torch.max(respective_im_0_plocss[idx][:,1]).item()) + ")") # quick way of getting positions of current image; TODO: get these from the image itself?
    # print total zero count on grid
    zero_mask = shear_grid == 0
    zero_count = np.count_nonzero(zero_mask)
    print("Grid Zero Count: ", zero_count, "out of", grid_size[0]*grid_size[1],"(" + str(100*zero_count/(grid_size[0]*grid_size[1]))+"%)")

    if image_overlay:
        plt.imshow(image)

    if show_zeros: # show zero values as black 
        masked_shear_grid = np.ma.masked_where(zero_mask, shear_grid)
        cmap = matplotlib.cm.coolwarm
        cmap.set_bad(color='black', alpha=0.3)
        plt.imshow(masked_shear_grid, cmap=cmap, vmin = np.min(shear_grid), vmax = np.max(shear_grid), extent=(0, original_size[0], 0, original_size[1]))
    else:
        plt.imshow(shear_grid, cmap="coolwarm", vmin = np.min(shear_grid), vmax = np.max(shear_grid), extent=(0, original_size[0], 0, original_size[1]))
    plt.colorbar(label=shear_type)
    plt.title(shear_type)
    plt.grid(True)
    plt.show()

In [ ]:
# scale is 10 pixels : 1 point on graph (as defined below), change if necessary
grid_size = (400, 400)  # Define the size of the grid
original_size = (4000, 4000)
shear_type = "Horizontal Shear"
for idx in range(4): #98 patches in total, this is just the first few
    plot_single_shear_map(idx, grid_size, original_size, shear_type, show_zeros=False, image_overlay=True) # change show_zeros to visualize the maps alone

In [ ]:
# scale is 10x10 pixels : 1 point on graph (as defined below)
grid_size = (400, 400)  # Define the size of the grid
original_size = (4000, 4000)
shear_type = "Diagonal Shear"
for idx in range(4): #98 patches in total, this is just the first few
    plot_single_shear_map(idx, grid_size, original_size, shear_type, show_zeros=False) # change or remove show_zeros to visualize the maps alone